# Tier 3 Assignment Set 4: New Topics (19–23)

---

These exercises cover five advanced bioinformatics topics:
- **Topic 19 — GWAS:** Chi-square association tests and genomic inflation
- **Topic 20 — Spatial Transcriptomics:** Neighbor graphs and Moran's I autocorrelation
- **Topic 21 — Copy Number Analysis:** Log-ratio segmentation
- **Topic 22 — Bayesian Statistics:** Sequential Bayesian updating and Bayes factors
- **Topic 23 — TF Footprinting & Chromatin Accessibility:** ATAC-seq fragment classification and footprint scores

---

## Grading Rubric

| Problem | Topic | Stars | Points |
|---------|-------|-------|--------|
| 1 | GWAS Chi-Square Association Test | ★ | 10 |
| 2 | Genomic Inflation Factor | ★★ | 10 |
| 3 | Spatial Neighbor Graph | ★ | 10 |
| 4 | Moran's I Spatial Autocorrelation | ★★ | 15 |
| 5 | Log-Ratio Copy Number Segmentation | ★★ | 15 |
| 6 | Bayes Factor for Variant Pathogenicity | ★★ | 15 |
| 7 | ATAC-seq Fragment Size Classification | ★ | 15 |
| 8 | TF Footprint Score | ★★★ | 10 |
| **Total** | | | **100** |

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
import numpy as np
import math
import random

---

## Problem 1: GWAS Chi-Square Association Test (★, 10 pts)

Given a 2×2 contingency table of allele counts `[[case_ref, case_alt], [ctrl_ref, ctrl_alt]]`,
compute the following statistics **without** using `scipy.stats`:

- **Chi-square statistic** using the exact formula:  
  χ² = N(ad − bc)² / [(a+b)(c+d)(a+c)(b+d)]  
  where a = case_ref, b = case_alt, c = ctrl_ref, d = ctrl_alt, N = a+b+c+d
- **Alt allele frequency in cases:** `case_alt / (case_ref + case_alt)`
- **Alt allele frequency in controls:** `ctrl_alt / (ctrl_ref + ctrl_alt)`
- **Odds ratio:** `(case_alt * ctrl_ref) / (case_ref * ctrl_alt)`

Return a dict with keys `chi2`, `alt_freq_cases`, `alt_freq_controls`, `odds_ratio`,
all rounded to 4 decimal places.

**Grading rubric:**
- Correct chi-square formula using a, b, c, d notation (4 pts)
- Correct allele frequency calculations (3 pts)
- Correct odds ratio (3 pts)

In [ ]:
def gwas_chi2_test(contingency_table: list[list[int]]) -> dict:
    """
    Compute GWAS association statistics from a 2x2 allele count table.

    Args:
        contingency_table: [[case_ref, case_alt], [ctrl_ref, ctrl_alt]]

    Returns:
        Dict with keys 'chi2', 'alt_freq_cases', 'alt_freq_controls', 'odds_ratio',
        all rounded to 4 decimal places.
    """
    # YOUR CODE HERE
    pass


# Test
table = [[480, 120], [450, 50]]
result = gwas_chi2_test(table)
print(result)

---

## Problem 2: Genomic Inflation Factor (★★, 10 pts)

The genomic inflation factor λ detects systematic bias in GWAS results.
Given a list of chi-square statistics (each with 1 degree of freedom), compute:

1. **Implement median from scratch** (no `statistics` or `numpy.median`).
2. **λ = median(chi2_values) / 0.4549**  
   (0.4549 is the median of χ²(1) under the null hypothesis)
3. **Genomic control correction:** divide every chi-square value by λ.

Return `{"lambda": float, "corrected_chi2": list}` where `lambda` is rounded to 4 dp
and values in `corrected_chi2` are rounded to 4 dp.

**Grading rubric:**
- Median computed from scratch without library functions (4 pts)
- Correct λ formula (3 pts)
- All chi-square values correctly divided by λ (3 pts)

In [ ]:
def genomic_inflation(chi2_values: list[float]) -> dict:
    """
    Compute the genomic inflation factor and apply genomic control correction.

    Args:
        chi2_values: List of chi-square statistics (1 df each) from a GWAS.

    Returns:
        Dict with keys:
            'lambda': genomic inflation factor (4 dp)
            'corrected_chi2': list of chi2 values divided by lambda (each 4 dp)
    """
    # YOUR CODE HERE
    pass


# Test: inflate chi2 values so lambda > 1
random.seed(42)
chi2_vals = [random.uniform(0.1, 4.0) * 1.3 for _ in range(20)]
result = genomic_inflation(chi2_vals)
print(f"Lambda: {result['lambda']}")
print(f"First 5 corrected chi2: {result['corrected_chi2'][:5]}")

---

## Problem 3: Spatial Neighbor Graph (★, 10 pts)

Given a list of spatial spot coordinates `[(x, y), ...]` and a radius `r`, build a
neighbor graph: two spots are neighbors if their **Euclidean distance ≤ r**.

Return:
- `"adjacency"`: dict mapping each spot index to a sorted list of neighbor indices
  (excluding the spot itself)
- `"avg_neighbors"`: mean number of neighbors per spot (float, 4 dp)

**Grading rubric:**
- Correct Euclidean distance calculation (3 pts)
- Self-loops excluded from adjacency (3 pts)
- Correct average neighbor count (4 pts)

In [ ]:
def spatial_neighbor_graph(coords: list[tuple[float, float]], r: float) -> dict:
    """
    Build a spatial neighbor graph based on Euclidean distance.

    Args:
        coords: List of (x, y) coordinate tuples for each spot.
        r: Radius threshold; spots within this distance are neighbors.

    Returns:
        Dict with keys:
            'adjacency': {spot_index: [sorted neighbor indices]}
            'avg_neighbors': float (4 dp)
    """
    # YOUR CODE HERE
    pass


# Test
coords = [(0, 0), (1, 0), (0, 1), (5, 5), (5.5, 5)]
result = spatial_neighbor_graph(coords, r=1.5)
print("Adjacency:", result['adjacency'])
print("Avg neighbors:", result['avg_neighbors'])

---

## Problem 4: Spatial Autocorrelation — Moran's I (★★, 15 pts)

Moran's I measures whether nearby spots have similar expression values.
Given expression values and an adjacency list (from Problem 3 or provided), compute:

**I = (N / W) × (Σᵢ Σⱼ wᵢⱼ (xᵢ − x̄)(xⱼ − x̄)) / Σᵢ(xᵢ − x̄)²**

where:
- N = number of spots
- W = total weight (sum of all neighbor counts across all spots)
- wᵢⱼ = 1 if j is in adjacency[i], else 0
- x̄ = mean expression

Positive I → spatial clustering; negative I → spatial dispersion.
Return Moran's I as a float rounded to 4 dp.

**Grading rubric:**
- Correct computation of W (total edges) (3 pts)
- Correct numerator cross-product sum (7 pts)
- Correct denominator (variance term) and final ratio (5 pts)

In [ ]:
def morans_i(expression: list[float], adjacency: dict) -> float:
    """
    Compute Moran's I spatial autocorrelation statistic.

    Args:
        expression: List of expression values, one per spot.
        adjacency: Dict mapping spot index to list of neighbor indices.

    Returns:
        Moran's I statistic (float, 4 dp).
    """
    # YOUR CODE HERE
    pass


# Test: strong spatial clustering — neighbors have similar high/low values
expr = [10.0, 10.5, 9.8, 1.0, 1.2]
adj  = {0: [1, 2], 1: [0, 2], 2: [0, 1], 3: [4], 4: [3]}
i_stat = morans_i(expr, adj)
print(f"Moran's I: {i_stat}  (expect positive, near 1)")

---

## Problem 5: Log-Ratio Copy Number Segmentation (★★, 15 pts)

Implement a simplified CBS (Circular Binary Segmentation)-like algorithm for
copy-number segmentation:

1. Compute a **sliding window mean** with `window=5` (use only full windows;
   the first smoothed value corresponds to the centre of the first full window).
2. Find **breakpoints** where the absolute difference between consecutive
   smoothed values exceeds `threshold` (default 0.3).
3. Convert breakpoint positions back to bin indices (accounting for the window
   offset), then assign each segment the **mean of its original bins**.

Return a list of `(start_idx, end_idx, segment_mean)` tuples where indices are
0-based and inclusive, and `segment_mean` is rounded to 4 dp.

**Grading rubric:**
- Correct sliding window mean over full windows only (5 pts)
- Correct detection of breakpoints by threshold comparison (5 pts)
- Correct segment mean computation from original bins (5 pts)

In [ ]:
def segment_copy_numbers(
    log2_ratios: list[float],
    window: int = 5,
    threshold: float = 0.3,
) -> list[tuple[int, int, float]]:
    """
    Segment log2 copy-number ratios using a sliding-window CBS-like approach.

    Args:
        log2_ratios: List of log2 copy-number ratios, one per genomic bin.
        window: Sliding window size for smoothing.
        threshold: Minimum absolute difference between consecutive smoothed
                   values to call a breakpoint.

    Returns:
        List of (start_idx, end_idx, segment_mean) tuples.
        Indices are 0-based and inclusive; segment_mean is rounded to 4 dp.
    """
    # YOUR CODE HERE
    pass


# Test: flat region, then gain, then loss
ratios = [0.0] * 10 + [1.2] * 10 + [-0.8] * 10
segments = segment_copy_numbers(ratios)
for seg in segments:
    print(f"  [{seg[0]}:{seg[1]}]  mean={seg[2]}")

---

## Problem 6: Bayes Factor for Variant Pathogenicity (★★, 15 pts)

Implement **sequential Bayesian updating** to estimate the posterior probability
that a variant is pathogenic after observing a series of test results.

**Update rules** (apply once per test result in order):
- For a **positive** result:  
  P(path | pos) = sens × P_path / [sens × P_path + (1 − spec) × (1 − P_path)]
- For a **negative** result:  
  P(path | neg) = (1 − sens) × P_path / [(1 − sens) × P_path + spec × (1 − P_path)]

Return the **final posterior probability** as a float rounded to 4 dp.

**Grading rubric:**
- Correct positive-result update formula (5 pts)
- Correct negative-result update formula (5 pts)
- Sequential application across all results (5 pts)

In [ ]:
def bayesian_pathogenicity(
    prior: float,
    sensitivity: float,
    specificity: float,
    results: list[str],
) -> float:
    """
    Sequentially update the probability of variant pathogenicity.

    Args:
        prior: Prior probability the variant is pathogenic (0–1).
        sensitivity: P(positive test | pathogenic).
        specificity: P(negative test | benign).
        results: Ordered list of test outcomes: 'positive' or 'negative'.

    Returns:
        Final posterior probability of pathogenicity (float, 4 dp).
    """
    # YOUR CODE HERE
    pass


# Test
posterior = bayesian_pathogenicity(
    prior=0.1,
    sensitivity=0.9,
    specificity=0.95,
    results=['positive', 'positive', 'negative'],
)
print(f"Final posterior: {posterior}")

---

## Problem 7: ATAC-seq Fragment Size Classification (★, 15 pts)

In ATAC-seq, fragment lengths reflect nucleosome occupancy.
Classify each fragment by length:

| Category | Length range |
|---|---|
| Sub-nucleosomal (NFR) | < 147 bp |
| Mono-nucleosomal | 147–294 bp |
| Di-nucleosomal | 295–441 bp |
| Tri-nucleosomal | ≥ 442 bp |

Also compute the **NFR ratio** = sub_nucleosomal / total.

Return a dict with keys `sub_nuc`, `mono_nuc`, `di_nuc`, `tri_nuc` (ints),
`nfr_ratio` (float, 4 dp), and `total` (int).

**Grading rubric:**
- Correct boundary conditions for all four categories (8 pts)
- Correct NFR ratio (4 pts)
- Correct total count (3 pts)

In [ ]:
def classify_atac_fragments(fragment_lengths: list[int]) -> dict:
    """
    Classify ATAC-seq fragment lengths into nucleosomal categories.

    Args:
        fragment_lengths: List of fragment lengths in base pairs.

    Returns:
        Dict with keys 'sub_nuc', 'mono_nuc', 'di_nuc', 'tri_nuc' (int counts),
        'nfr_ratio' (float, 4 dp), and 'total' (int).
    """
    # YOUR CODE HERE
    pass


# Test
random.seed(7)
fragments = (
    [random.randint(50, 146) for _ in range(50)]   # sub-nuc
    + [random.randint(147, 294) for _ in range(30)] # mono-nuc
    + [random.randint(295, 441) for _ in range(15)] # di-nuc
    + [random.randint(442, 600) for _ in range(5)]  # tri-nuc
)
result = classify_atac_fragments(fragments)
print(result)

---

## Problem 8: TF Footprint Score (★★★, 10 pts)

TF footprints appear as dips in Tn5 insertion profiles centred on bound motifs.
Given a 2D array of Tn5 insertion counts `(n_sites × n_positions)` centred on
TF motif occurrences:

1. Average across sites to get a 1D profile.
2. Find `center = n_positions // 2`.
3. Define regions (using `flank_half=10` by default):
   - **central:** `profile[center - flank_half : center + flank_half]`
   - **left flank:** `profile[center - 3*flank_half : center - flank_half]`
   - **right flank:** `profile[center + flank_half : center + 3*flank_half]`
4. **footprint_score = mean(left_flank + right_flank) − mean(central)**  
   (positive score = genuine footprint / depletion in centre)

Return `{"footprint_score": float (4 dp), "avg_profile": list (4 dp each)}`.

**Grading rubric:**
- Correct site-averaging to produce 1D profile (3 pts)
- Correct central and flanking region slices (4 pts)
- Correct footprint score formula (3 pts)

In [ ]:
def tf_footprint_score(
    insertion_profiles: list[list[float]],
    flank_half: int = 10,
) -> dict:
    """
    Compute a TF footprint score from per-site Tn5 insertion profiles.

    Args:
        insertion_profiles: 2D array of shape (n_sites, n_positions).
                            Each row is the insertion count profile for one site.
        flank_half: Half-width of the central footprint region and each
                    flanking window (default 10).

    Returns:
        Dict with keys:
            'footprint_score': float (4 dp)
            'avg_profile': list of floats (4 dp each)
    """
    # YOUR CODE HERE
    pass


# Test: simulate a clear footprint (low centre, high flanks)
np.random.seed(0)
n_sites, n_pos = 50, 100
profiles = np.random.poisson(lam=5, size=(n_sites, n_pos)).astype(float)
# Deplete centre ±10 around position 50
profiles[:, 40:60] *= 0.2
result = tf_footprint_score(profiles.tolist())
print(f"Footprint score: {result['footprint_score']}  (expect positive)")
print(f"Avg profile length: {len(result['avg_profile'])}")

---

## Summary

In this assignment you worked with five advanced bioinformatics topics:

1. **GWAS Chi-Square Test**: 2×2 allele count tables, odds ratios, and allele frequencies
2. **Genomic Inflation Factor**: Median-based λ and genomic control correction
3. **Spatial Neighbor Graph**: Euclidean distance thresholding over spot coordinates
4. **Moran's I**: Spatial autocorrelation quantifying expression clustering
5. **Copy Number Segmentation**: Sliding-window smoothing and breakpoint detection
6. **Bayesian Pathogenicity**: Sequential posterior updating with sensitivity/specificity
7. **ATAC-seq Fragment Classification**: Nucleosomal size categories and NFR ratio
8. **TF Footprint Score**: Site-averaged Tn5 profiles and flanking vs. central depletion

---